In [1]:
import requests
import json
import random
import os
# ★ 주피터 노트북 전용 출력 제어 라이브러리 추가
from IPython.display import clear_output 

apikey = '4FFAF72BE3F45EA48551E923062FCA5E'
list_used = []  # 사용한 단어 저장 리스트
com_word = ""   # 컴퓨터가 마지막으로 말한 단어 저장 변수
msg_error = ""  # 에러 메시지 저장

def clear_screen():
    """주피터 노트북의 셀 출력부를 깨끗하게 지움"""
    # wait=True는 다음 출력이 올 때까지 기다렸다가 지워주어 화면 깜빡임을 줄여줍니다.
    clear_output(wait=True)

def get_dueum(word):
    """단어의 마지막 글자에 대한 두음법칙 적용 글자 반환"""
    if not word: return ""
    char = word[-1]
    code = ord(char) - 44032
    if code < 0 or code > 11171: return char
    cho = (code // 28) // 21
    jung = (code // 28) % 21
    jong = code % 28
    if cho == 5:
        if jung in [2, 4, 7, 12, 17, 20]:
            return chr(44032 + (11 * 21 + jung) * 28 + jong)
        else:
            return chr(44032 + (2 * 21 + jung) * 28 + jong)
    elif cho == 2:
        if jung in [4, 12, 17, 20]:
            return chr(44032 + (11 * 21 + jung) * 28 + jong)
    return char

def display_game():
    """현재 게임 상태를 주피터 셀에 상시 고정 출력"""
    clear_screen()
    print("="*45)
    print("        [ 🤖 AI 끝말잇기 - Jupyter Mode ]")
    print("="*45)
    
    if list_used:
        print("\n📜 현재 사용된 단어 리스트")
        for i in range(0, len(list_used), 10):
            line = list_used[i:i+10]
            print(f"[{i+1:02d}~{i+len(line):02d}]: {', '.join(line)}")
        print("-" * 45)
    
    if com_word:
        dueum = get_dueum(com_word)
        target = f"'{com_word[-1]}'" if com_word[-1] == dueum else f"'{com_word[-1]}' 또는 '{dueum}'"
        print(f"\n💬 컴퓨터: {com_word}")
        print(f"👉 {target}(으)로 시작하는 2글자 이상의 단어를 입력하세요.")
    else:
        print("\n🚀 첫 단어를 입력하여 게임을 시작하세요! (2글자 이상)")

# 메인 게임 루프
while True:
    while True:
        display_game()
        if msg_error:
            print(f"\n⚠️  경고: {msg_error}")
            msg_error = "" # 출력 후 초기화
            
        player = input('\n입력창 ⌨️ : ').strip()
        
        if player == '그만':
            break
            
        # [규칙 1] 두 글자 이상 제한
        if len(player) < 2:
            msg_error = "두 글자 이상의 단어만 입력할 수 있습니다."
            continue

        # [규칙 2] 끝말잇기 성립 여부 (두음법칙 포함)
        if com_word != "":
            dueum_target = get_dueum(com_word)
            if player[0] != com_word[-1] and player[0] != dueum_target:
                msg_error = f"'{com_word[-1]}'(으)로 시작해야 합니다."
                continue 
        
        # [규칙 3] 중복 확인
        if player in list_used:
            msg_error = f"'{player}'은(는) 이미 사용한 단어입니다."
            continue

        # [규칙 4] 사전 확인
        check_url = f'https://opendict.korean.go.kr/api/search?certkey_no=8917&key={apikey}&target_type=search&req_type=json&part=word&q={player}'
        is_valid = False
        try:
            check_res = requests.get(check_url)
            check_data = json.loads(check_res.text)
            if 'item' in check_data['channel']:
                for item in check_data['channel']['item']:
                    target = item['word'].replace('-', '').replace('^', '')
                    if target == player:
                        is_valid = True
                        break
            if not is_valid:
                msg_error = f"'{player}'은(는) 사전에 없는 단어입니다."
                continue
        except:
            msg_error = "네트워크 연결을 확인해주세요."
            continue
        break
    
    if player == '그만':
        print("\n👋 게임을 종료합니다. 수고하셨습니다!")
        break
        
    list_used.append(player)
    
    # 컴퓨터 답변 로직
    search_chars = list(set([player[-1], get_dueum(player)]))
    list_a = []
    for query in search_chars:
        url = f'https://opendict.korean.go.kr/api/search?certkey_no=8917&key={apikey}&target_type=search&req_type=json&part=word&sort=popular&start=1&num=50&pos=1&type3=general&q={query}'
        try:
            response = requests.get(url)
            words = json.loads(response.text)
            if 'item' in words['channel']:
                for item in words['channel']['item']:
                    target_word = item.get('sense', item).get('word').replace("-", "").replace("^", "")
                    if len(target_word) >= 2 and target_word[0] == query:
                        list_a.append(target_word)
        except:
            continue

    candidates = list(set(list_a) - set(list_used))
    if candidates:
        com_word = random.choice(candidates)
        list_used.append(com_word)
    else:
        display_game()
        print("\n🎊 축하합니다! 컴퓨터가 패배했습니다!")
        input("\n(엔터를 누르면 새 게임이 시작됩니다)")
        list_used = []
        com_word = ""

        [ 🤖 AI 끝말잇기 - Jupyter Mode ]

📜 현재 사용된 단어 리스트
[01~10]: 끝말잇기, 기우, 우비, 비슴, 슴슴하다, 다만, 만조, 조가딸, 딸기, 기억
[11~16]: 억장, 장용, 용수, 수반되다, 다뇨, 요리
---------------------------------------------

💬 컴퓨터: 요리
👉 '리' 또는 '이'(으)로 시작하는 2글자 이상의 단어를 입력하세요.

👋 게임을 종료합니다. 수고하셨습니다!
